# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreyaagenixx/flyrank-intern-task-assignment1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
#ML Task Type: Supervised Binary Classification & Learning-to-Rank (Scoring)Explanation: While the end goal is to output a ranked queue of pages that need updating, the core prediction task is binary classification. For every unique URL, the model scores the probability ($0.0$ to $1.0$) that the page will experience structural organic search degradation over the next observation window. We then rank by this calculated probability to build our final task list.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
 #Target Label: is_declining_label (Binary: 1 for severe degradation, 0 for stable/growing).

#How it is constructed: Because true "content obsolescence" isn't a single column logged in a database, we use a calculated proxy label. A page is marked as 1 (declining) if its rolling 90-day search click-through trend exhibits a statistically significant negative slope combined with an average search position drop greater than 3 positions compared to its historical 365-day baseline, stripped of holiday noise.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# Primary Evaluation Metric: Precision@K (specifically Precision@50).Business Justification: The editorial team only has the bandwidth to rewrite 50 pages per week. If we use a metric like standard accuracy or ROC-AUC, we might optimize for correctly identifying thousands of healthy pages (0s) while our top recommendations remain noisy. Precision@50 measures exactly what matters: out of the top 50 pages we tell the writers to fix, how many are actually degrading? Our target threshold is to beat the human baseline ($24\%$) by hitting at least $60\%$ precision.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import os
import sys
import subprocess
import pandas as pd

# 1. Environment & Path Alignment Setup
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

# 2. Load and inspect the unit of analysis
data_path = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

# Map human-readable column for binary label validation
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print("=== UNIT OF ANALYSIS DATAFRAME ===")
print(f"One row represents: ONE UNIQUE PAGE (URL) for a specific tracking period.\n")
print(f"Total Rows (Samples): {df.shape[0]}")
print(f"Total Columns (Features + Target): {df.shape[1]}")
print("\n--- Example Feature Matrix & Target Slice ---")
display(df[['client_id', 'content_age_days', 'impressions_90d', 'avg_position', 'ctr', 'is_declining_label']].head())

print("\n--- Class Distribution of Target Variable ---")
print(df['is_declining_label'].value_counts(normalize=True).rename({0: 'Stable/Up (0)', 1: 'Declining (1)'}))

=== UNIT OF ANALYSIS DATAFRAME ===
One row represents: ONE UNIQUE PAGE (URL) for a specific tracking period.

Total Rows (Samples): 30000
Total Columns (Features + Target): 45

--- Example Feature Matrix & Target Slice ---


,client_id,content_age_days,impressions_90d,avg_position,ctr,is_declining_label
0,client_f369cb89fc,187,3803,10.6,0.76,1
1,client_4e07408562,445,15320,20.3,0.05,1
2,client_7f2253d7e2,141,12581,36.5,0.09,1
3,client_19581e27de,463,11751,6.2,0.49,0
4,client_3fdba35f04,263,19140,44.0,0.13,1



--- Class Distribution of Target Variable ---
is_declining_label
Declining (1)    0.542067
Stable/Up (0)    0.457933
Name: proportion, dtype: float64


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
#Limitations of a static rule: A typical human engineering rule would be: "Flag any page where clicks dropped more than 15% month-over-month." This fails catastrophically for three reasons:

#Seasonality: An outdoor gear guide will drop 50% in traffic every October naturally; a static rule triggers a false positive rewrite.

#Macro Site Fluctuations: If the entire domain loses a core update, all pages drop. A static rule flags everything instead of finding the specific pages decaying due to stale content.

#Non-linear interactions: Traffic drop alone doesn't isolate the issue. ML can weigh high impressions combined with collapsing click-through-rates (CTR) and creeping average position simultaneously.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.